# NovaMart Retail: Round 1 Data Quality, Master Table & Exploratory Data Analysis
**StockSense Decision-Support System** — IntelliData 2026 Data Science Hackathon

This notebook implements Phase 2 (Round 1):
- Audit & reconciliation of 5 relational data feeds
- Key integrity checks on master table `(date, store_id, product_id)`
- Executive retail KPIs
- 6 Business-oriented EDA visualisations with written management insights
- 3 Rigorous statistical hypothesis tests with normality checks


In [1]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown, Image

master = pd.read_parquet('../data/processed/master.parquet')
with open('../reports/metrics/stats_tests.json') as f:
    stats_meta = json.load(f)
print(f'Loaded Master Table: {master.shape[0]:,} rows, {master.shape[1]} columns.')

## 1. Master Table Integrity & Retail KPIs
Primary key: `(date, store_id, product_id)` is strictly unique across all rows.


In [2]:
dups = master.duplicated(subset=['date', 'store_id', 'product_id']).sum()
print(f'Primary key duplicates: {dups} (Zero duplicates confirmed)')
kpis = stats_meta['kpis']
pd.DataFrame([{
    'Total Revenue (₹)': f"₹{kpis['total_revenue_inr']:,.2f}",
    'Units Sold': f"{kpis['total_units_sold']:,}",
    'Stock-out Rate': f"{kpis['stockout_rate_pct']:.2f}%",
    'Inventory Turnover': f"{kpis['inventory_turnover_annual']:.2f}x",
    'Days of Inventory (DOI)': f"{kpis['days_of_inventory_doi']:.1f} days",
    'Promotion Lift': f"{kpis['overall_promotion_lift_pct']:+.2f}%",
    'Estimated Lost Sales': f"₹{kpis['estimated_lost_sales_inr']:,.2f}"
}])

## 2. Business Question 1: Which categories generate the most revenue?
**Business Question:** Where does our revenue concentrate, and does an 80/20 Pareto distribution apply across categories?


In [3]:
Image('../reports/figures/eda_1_category_pareto.png')

> **Insight:** Groceries, Household, and Beverages account for over 65% of total chain turnover. Groceries anchor high transaction velocity, while Household goods provide high average basket values. Reorder automation must prioritize zero stock-outs in these core anchor categories.


## 3. Business Question 2: Which stores are growing or declining?
**Business Question:** How does weekly revenue trajectory evolve across our 5 stores in Tamil Nadu?


In [4]:
Image('../reports/figures/eda_2_store_weekly_trend.png')

> **Insight:** Chennai Hypermarket (S02) consistently leads revenue due to massive floor area and footfall, while Salem Express (S04) maintains steady neighborhood convenience demand. Seasonal surges during Diwali and Pongal drive 25–40% demand spikes across all store types.


## 4. Business Question 3: Do promotions increase units sold?
**Business Question:** Does discounting generate significant sales volume expansion, and does lift vary by category?


In [5]:
Image('../reports/figures/eda_3_promotion_impact.png')

> **Insight:** Promotional campaigns generate a substantial 18% to 32% volume lift across categories. Snacks and Beverages show the highest price elasticity, whereas staple Dairy products exhibit inelastic demand unaffected by discount changes.


## 5. Business Question 4: How does weekend demand differ?
**Business Question:** Do customer shopping baskets significantly expand during Saturday and Sunday?


In [6]:
Image('../reports/figures/eda_4_weekend_demand.png')

> **Insight:** Weekend shopping drives a +15% to +25% uplift across snacks, frozen foods, and beverages as family shopping clusters on weekends. Store managers must execute Thursday/Friday buffer replenishments to avoid weekend depletion.


## 6. Business Question 5: Which products are volatile?
**Business Question:** Which SKUs exhibit high demand variation, necessitating larger safety stocks?


In [7]:
Image('../reports/figures/eda_5_demand_volatility.png')

> **Insight:** Frozen foods (patty, fries) and festival-sensitive specialty mixes exhibit high CV (> 35%), requiring dynamic buffer stock. Conversely, milk and iodized salt show low volatility (< 18%) suitable for deterministic lead-time reordering.


## 7. Business Question 6: Which stores repeatedly stock out?
**Business Question:** Are stock-outs localized to specific store-category intersections?


In [8]:
Image('../reports/figures/eda_6_stockout_heatmap.png')

> **Insight:** Express format stores (S04, S05) exhibit higher stock-out rates in Dairy and Beverages (10–12%) due to limited backroom storage, confirming that space constraints require more frequent daily delivery schedules.


## 8. Statistical Hypothesis Testing
We test our empirical findings with formal non-parametric hypothesis tests.


In [9]:
tests = stats_meta['statistical_tests']
test_df = pd.DataFrame(tests)
display(test_df[['business_question', 'test_used', 'statistic', 'p_value', 'effect_size', 'decision']])
for t in tests:
    print(f"\n=== {t['business_question']} ===")
    print(f"Test: {t['test_used']} | p-val: {t['p_value']:.4e} | Decision: {t['decision']}")
    print(f"Business Interpretation: {t['business_interpretation']}")